# Lab 5

Integrantes:
- Francis Aguilar, 22243
- César López, 22535
- Jose Marchena, 22398

Repo:
https://github.com/MarchMol/lab6_vision

Imagina que trabaja en una startup de AgriTech en Guatemala. La empresa está desarrollando una
aplicación móvil para ayudar a pequeños agricultores a detectar enfermedades en sus cultivos de mango de
forma temprana. Los agricultores utilizarán teléfonos inteligentes de gama baja (Android) y, debido a la
ubicación remota de las fincas, la aplicación debe funcionar 100% offline (Edge AI), haciendo la inferencia
en el dispositivo sin enviar imágenes a la nube.
Para este laboratorio usará el dataset proporcionado.
Dataset: Mango Leaf Disease Dataset (Kaggle) [Sí, el mismo del examen corto 1]

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aryashah2k/mango-leaf-disease-dataset")

print("Path to dataset files:", path)

## Task 1
Como Ingeniero Principal (Lead AI Engineer) del proyecto AgriTech, usted debe justificar las decisiones
arquitectónicas ante su equipo y sus clientes. Responda a los siguientes escenarios en su reporte (máximo
1 página por respuesta), combinando la teoría matemática con el pragmatismo laboral.


1. Un desarrollador junior de su equipo sugiere: "Para detectar con mayor precisión las texturas de las hojas enfermas, deberíamos construir una red secuencial clásica (tipo VGG) pero de 150 capas. Más profundo siempre es mejor". Como líder técnico, explíquele argumentativamente por qué esta red fracasará estrepitosamente en el entrenamiento (mencionando el fenómeno de degradación y el desvanecimiento del gradiente). Luego, justifique cómo la adición estructural de las conexiones residuales (𝐹(𝑥) + 𝑥) de ResNet rescata el proyecto, haciendo viable entrenar redes ultra-profundas sin colapsar.


Por más que la idea de "mientras más profundo mejor" suene bien, pero en VGG no funciona del todo bien. En este caso al aplicar 150 capas se crea una red muy profunda y en la fase de backpropagation el gradiente se va multiplicando capa tras capa. Si los valores son menores que 1 el gradiente se vuelve extremadamente pequeño en las primeras capas y esas capas dejan prácticamente de aprender. Las capas iniciales se congelan y no terminan de converger bien. 

Sucede el fenómeno de degradación, que es un problema más crítico. Este problema se manifiesta cuando una red más profunda no solo deja de mejorar, sino que incluso presenta un mayor error de entrenamiento que una red más superficial. En teoría una red de 150 capas debería poder al menos igualar el rendimiento de una red más pequeña, ya que podría aprender funciones identidad en las capas adicionales, pero lo que sucede es que los algoritmos de optimización no logran encontrar estas soluciones, lo que lleva a un empeoramiento del rendimiento.

Las redes residuales (ResNet) abordan estos problemas mediante la introducción de conexiones de salto o skip connections, que permiten reformular el aprendizaje. En lugar de obligar a cada bloque de capas a aprender una transformación completa H(x), se le pide que aprenda una función residual F(x), de modo que la salida final sea F(x)+x. Esta modificación aparentemente simple tiene un impacto profundo, ya que facilita tanto la propagación de la información hacia adelante como la del gradiente hacia atrás. Gracias a esto las capas extra dejan de ser un obstáculo y pasan a ser opcionales desde el punto de vista funcional.

mientras una VGG de 150 capas colapsaría durante el entrenamiento, una arquitectura basada en ResNet no solo es viable, sino que puede aprovechar la profundidad para mejorar significativamente la extracción de características complejas, como las texturas finas en hojas enfermas.



2. Las enfermedades en las hojas de mango son visualmente heterogéneas: La Antracnosis se presenta como puntos negros diminutos, mientras que el Moho Polvoriento cubre áreas enormes de la hoja. Analice cómo la topología en paralelo del módulo Inception (usando filtros 3x3 y 5x5 simultáneamente) es ideal para este problema biológico en particular. Además, desde una perspectiva de costos de infraestructura (uso de GPUs en AWS o Google Cloud), explique cómo la inserción estratégica de convoluciones de 1x1 evita la explosión de la dimensionalidad y salva el presupuesto mensual de la startup.

Presentan patrones visuales muy distintos en escala y forma, la antracnosis aparece como puntos negros pequeños y localizados, mientras que el moho polvoriento se manifiesta como regiones extensas y difusas. Este tipo de variabilidad hace que una arquitectura tradicional, que procesa la imagen con un único tamaño de filtro por capa, sea limitada. En cambio, el módulo Inception permite aplicar simultáneamente filtros de distintos tamaños sobre la misma entrada. Esto significa que la red puede capturar en una sola etapa tanto detalles finos como patrones más globales, sin tener que depender de una sola escala de análisis.

Esto es especialmente útil en problemas biológicos, donde todo es más irregular y menos predecible que en otros contextos. Una misma hoja puede tener manchas pequeñas y zonas grandes afectadas al mismo tiempo, y el modelo necesita detectar ambas cosas sin perder información. Gracias a esta estructura en paralelo, Inception logra construir una representación más completa y flexible, lo que mejora bastante la precisión al diferenciar enfermedades.

Ahora, el problema es que todo eso puede volverse muy caro computacionalmente si no se controla. Ahí es donde entran las convoluciones de 1x1, que básicamente reducen la cantidad de información antes de aplicar los filtros más pesados. Esto hace que el modelo use menos memoria y menos tiempo de GPU. En servicios como AWS o Google Cloud, eso se traduce directamente en menos dinero gastado, así que estas capas no solo ayudan técnicamente, sino que también mantienen el proyecto dentro de un presupuesto razonable.

3. El cliente final (el agricultor) usará la aplicación en un teléfono Android de gama baja en medio del campo, sin conexión a internet. Sabemos que MobileNet logra esta eficiencia gracias a la Depthwise Separable Convolution. Describa brevemente cómo esta convolución divide el trabajo (filtrado espacial vs. combinación de canales). Sin embargo, en ingeniería no hay soluciones mágicas, todo tiene un precio: ¿Qué capacidad matemática o expresividad de la red se está sacrificando al separar estos dos pasos en comparación con una convolución estándar? Analice críticamente por qué, como director del proyecto, usted acepta pagar ese "costo" en este escenario comercial.

En MobileNet, la Depthwise Separable Convolution divide el trabajo de una convolución tradicional en dos pasos mucho más livianos. Primero, la depthwise convolution aplica un filtro espacial (por ejemplo, 3x3) por cada canal de entrada de forma independiente**, capturando patrones como bordes o texturas dentro de cada canal. Luego, la pointwise convolution (un filtro 1x1) se encarga de combinar la información entre canales, es decir, mezcla lo que cada canal detectó para construir características más completas. En lugar de hacer todo en una sola operación costosa, separa el “ver detalles” del “mezclar información”.

Pero esta eficiencia tiene un costo. En una convolución estándar, cada filtro aprende simultáneamente relaciones espaciales y entre canales, lo que le da más capacidad de representación. Al separar estos pasos, se pierde cierta capacidad para modelar interacciones complejas entre canales y espacio al mismo tiempo. En otras palabras, la red se vuelve un poco menos expresiva: puede necesitar más capas o puede no capturar patrones muy sutiles con la misma facilidad que una red más pesada.

Aun así, como director del proyecto, aceptar ese costo es totalmente razonable. El modelo se va a ejecutar en un teléfono Android de gama baja, sin internet y con recursos muy limitados. En ese contexto, un modelo más “perfecto” pero pesado simplemente no funcionaría: sería lento, consumiría demasiada batería o ni siquiera correría. MobileNet, aunque sacrifica algo de precisión teórica, ofrece un equilibrio mucho más valioso: suficiente desempeño con una eficiencia que hace viable el producto en el mundo real. Al final, un modelo que funciona bien en el campo es mucho más útil que uno más preciso que nunca se puede usar.

## Task 2
Utilice PyTorch o TensorFlow/Keras (a su elección). Debe escribir el código desde cero o basarse en las
documentaciones oficiales. Ejecute sus experimentos en Google Colab, Kaggle Notebooks o en su GPU
local. Sientanse libres de hacer uso de IA de forma educada, es decir, entendiendo realmente que lo que
esté haciendo sea realmente lo que necesitan y que sobretodo lo entiendan.
1. Descargue el dataset, divídalo en Entrenamiento (70%), Validación (15%) y Prueba (15%). Implemente Data Augmentation (rotaciones, flips, recortes) vital para evitar el sobreajuste en imágenes agrícolas.


2. Cargue los siguientes modelos pre-entrenados en ImageNet y congele sus capas base, reemplazando solo el cabezal de clasificación para nuestras 8 clases de mango:

  - a. ResNet (Puede usar ResNet50).
  - b. Inception (Puede usar InceptionV3).
  - c. MobileNet (Puede usar MobileNetV2 o V3-Small/Large).
  
3. Entrene los 3 modelos utilizando la misma función de pérdida (Cross-Entropy) y optimizador (ej. Adam) por un máximo de 15 a 20 épocas (o use Early Stopping).

4. Registre las métricas correspondientes para cada modelo:
- a. Accuracy (Exactitud) en el conjunto de prueba.
- b. F1-Score (Macro) en el conjunto de prueba.
- c. Tamaño del modelo final (en Megabytes) al guardarlo en disco (.pth o .h5).
- d. Tiempo de Inferencia: Cuántos milisegundos (ms) tarda en predecir una sola imagen (promedio sobre 100 imágenes).

## Task 3
En la industria, el código es solo una herramienta; lo que el cliente paga es su criterio. Basado en los
resultados de la Parte 2, redacte un dictamen ejecutivo de 1 a 2 páginas.
1. Presente una tabla clara cruzando los 3 modelos versus las 4 métricas evaluadas (Accuracy, F1- Score, Tamaño en MB, Tiempo de Inferencia).


In [3]:
import json
import pandas as pd

with open("models_results.json", "r") as f:
    data = json.load(f)

cols = {
    "acc": "Accuracy",
    "f1": "F1-Score",
    "size_MB": "Tamaño en MB",
    "time": "Tiempo de Inferencia (ms)"
}
rows = []
for model in data.keys():
    entry = {"Model": model}
    for k, v in cols.items():
        entry[v] = data[model][k]
        
    rows.append(entry)
    
df = pd.DataFrame(rows)
df

,Model,Accuracy,F1-Score,Tamaño en MB,Tiempo de Inferencia (ms)
0,ResNet50,0.985000,0.984962,90.042741,6.817027
1,InceptionV3,0.893333,0.893560,96.189900,13.309868
2,MobileNetV2,0.988333,0.988321,8.756724,8.022233


2. Compare a ResNet e Inception frente a MobileNet. ¿Cuánto "Accuracy" sacrificó usted (si es quesacrificó algo) al usar MobileNet? ¿Cómo se correlaciona el tamaño en Megabytes con laarquitectura matemática que usted describió en la Parte 1?


**R//**
Sorprendentemente, Mobilenet fue el que mejor rindio de todos los modelos en terminos de accuracy sobrepasando ResNet por una fraccion de porcentaje. En todo caso, Inception fue el que le fue peor con una diferencia de casi 10% comparado con los otros dos modelos. Asimismo, no se obtuvo el tiempo de latencia que se esperaba pues ResNet fue el mas rapido a pesar de ser mas profundo que MobileNet.

En terminos de tamaño se obtuvieron resultados esperados. En terminos de la arquitectura matematica, los tamaños de ResNet e Inception fueron similares pues aplican mas o menos el mismo calculo para parametros, con un umbral esperado de O(K^2* C_in *C_out) de memoria, mientras que MobileNet gracias a sus optimizaciones con operaciones depthwise lo logra reducir a O(K * C_in * C_out). Esto se puede observar con la proporcion de casi 1|10 entre la memoria utilizada por cada uno

3. Responda al CEO de la startup: Considerando que los agricultores guatemaltecos usarán teléfonoscon 2GB de RAM sin internet, ¿qué modelo exacto mandamos a producción y por qué? (Justifiquepor qué el modelo ganador es viable para Edge AI frente a los perdedores).

**R//**
Para dispositivos mobiles, incluso el nombre lo indica, MobileNet es el indicado para el trabajo. Con su tamaño de 8MB esto no representa ni el 1% de la capacidad de memoria RAM de los dispositivos de la empresa lo que lo hace amigable en terminos de tamaño. Asimismo, un tiempo de inferencia de 8ms es bastante aceptable y sera casi instantaneo para los trabajdores.

Este modelo es el clasico para Edge computing, ha demostrado (y sigue demostrando) su relevancia en este campo con sus grandes optimizaciones y en nuestro caso fue el que incluso consigió mejor rendimiento en la clasificacion.

Realmente, no hay competicion entre modelos, MobileNet es el inicado para produccion sin duda alguna
